# Day 44: Support Vector Machines (SVM)

## The Idea
Support Vector Machines are a classification algorithm that, instead of just finding any boundary that separates classes, specifically looks for the boundary (a hyperplane) that has the widest possible margin - the largest distance - to the nearest points of each class. Those nearest points are called "support vectors," and they're the only points that actually determine where the boundary sits; every other point could move around freely without changing the result. This margin-maximizing approach tends to generalize well because it doesn't just barely separate the classes, it separates them with the most breathing room possible. SVMs work well on small-to-medium datasets and are especially known for handling non-linear boundaries via the "kernel trick" (covered a couple cells down). The cell below loads the classic Iris flower dataset (3 species, 4 numeric features) and splits it into train/test sets, which we'll use for every SVM variant in this notebook.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_iris(return_X_y=True)  # 150 samples, 4 features (petal/sepal length & width), 3 species classes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Linear Kernel
The "kernel" determines the shape of the decision boundary SVM can learn. A `linear` kernel restricts the boundary to a straight line (in 2D) or flat hyperplane (in higher dimensions), which is the simplest and fastest option. It's a good default choice when you have reason to believe the classes are roughly linearly separable, or as a quick baseline before trying anything fancier - it's also easier to interpret since the decision boundary is just a weighted sum of the features. The Iris dataset happens to be fairly linearly separable (especially between certain species pairs), so we'd expect this to perform reasonably well. The printed accuracy is simply the fraction of test-set flowers whose species the model predicted correctly.

In [ ]:
linear_svm = SVC(kernel='linear').fit(X_train, y_train)  # straight-line/hyperplane decision boundary
print('Linear kernel accuracy:', linear_svm.score(X_test, y_test))  # fraction of test samples classified correctly

## Non-linear Kernel (RBF)
Many real datasets aren't linearly separable - no straight line/plane can cleanly split the classes. The Radial Basis Function (RBF) kernel handles this by implicitly mapping the data into a much higher-dimensional space where a linear separator *would* work, without ever explicitly computing that transformation (this is the "kernel trick" - it computes similarities between points as if the transformation happened, which is far cheaper). The result is a decision boundary that can curve and wrap around clusters of points in the original feature space. `gamma='scale'` controls how far the influence of a single training point reaches: a small gamma means smooth, far-reaching influence (simpler boundary), while a large gamma means each point only influences its immediate neighborhood (more complex, potentially overfit boundary). Compare the accuracy here to the linear kernel's result above - if RBF does noticeably better, it's a sign the class boundaries have real curvature that a straight line can't capture.

In [ ]:
rbf_svm = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_train, y_train)  # RBF kernel allows curved decision boundaries
print('RBF kernel accuracy:', rbf_svm.score(X_test, y_test))

## Tuning C and Gamma
`C` and `gamma` are the two most important hyperparameters for an RBF-kernel SVM, and getting them right matters a lot for performance. `C` controls the tradeoff between a wide margin and correctly classifying every training point: a small `C` allows more misclassifications in exchange for a wider, smoother margin (more regularization, less overfitting), while a large `C` tries hard to classify every training point correctly even if that means a narrower, more convoluted margin (less regularization, more overfitting risk). `gamma` (mentioned above) controls the reach of each point's influence - small gamma = smoother/simpler boundary, large gamma = tighter/more complex boundary that can overfit. In practice both are tuned together via grid search or cross-validation rather than guessed. The loop below only varies C (gamma stays at its default) and prints test accuracy for each - watch whether accuracy keeps improving, plateaus, or gets worse as C grows, which tells you whether the model is under- or over-regularized.

In [ ]:
for C in [0.1, 1, 10]:  # smaller C = wider margin/more regularization; larger C = fits training points more tightly
    model = SVC(kernel='rbf', C=C).fit(X_train, y_train)
    print(f'C={C}: accuracy={model.score(X_test, y_test):.3f}')